# ML-09 — Validation and Research Claim Audit

**Lane:** Refresh / Content Opportunity Scoring

This notebook audits the Week-5 model using careful methodology, honest validation, leakage checks, real failure examples, and safe claim language.

**Before running:** replace `DATA_PATH`, `DATE_COLUMN`, `TARGET_COLUMN`, `FEATURES`, and `WEEK5_MAE` with the exact values from your Week-5 notebook. Do not invent metrics.


## 1. Two paper findings + my methodology questions

### Finding 1 — Content age and performance

The research paper examines relationships between content age and SEO performance.

**Methodology question:** Where does the outcome measure come from, and is the observation window consistent across pages? This matters because age can be associated with search demand, content type, authority, and update history.

I would treat an observed relationship as an association in the study data rather than proof that content age itself causes a performance change.

### Finding 2 — CTR and search performance

The paper reports findings involving click-through rate and search performance.

**Methodology question:** Does the comparison account for ranking position and other factors that affect both CTR and impressions? Ranking position can strongly influence CTR, so I would want to understand whether the validation/comparison design supports the strength of the conclusion.

These are constructive methodology questions, not judgments that the findings are right or wrong.


In [ ]:
paper_audit_items = [
    "Content age vs. performance: check outcome definition and observation window.",
    "CTR vs. search performance: check ranking-position effects and validation/comparison design."
]
for item in paper_audit_items:
    print("-", item)


## 2. My model under an honest split (before/after)

The Week-5 model is re-evaluated with a **time-aware split**: earlier observations are used for training and later observations for testing. This better represents a real forward-looking decision setting.

The comparison is a validation-design check. A more complex model is not automatically better.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -------- Replace these with your actual Week-5 values --------
DATA_PATH = "YOUR_DATASET.csv"
DATE_COLUMN = "date"
TARGET_COLUMN = "target_column"

FEATURES = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "position"
]

# Copy the real Week-5 MAE here.
WEEK5_MAE = None
# --------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

required = FEATURES + [DATE_COLUMN, TARGET_COLUMN]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing columns: {missing}. Update the configuration above."
    )

print("Dataset shape:", df.shape)
print("Required columns found.")


In [ ]:
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
df = df.dropna(subset=[DATE_COLUMN]).sort_values(DATE_COLUMN).reset_index(drop=True)

split_index = int(len(df) * 0.80)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train period:", train[DATE_COLUMN].min(), "to", train[DATE_COLUMN].max())
print("Test period:", test[DATE_COLUMN].min(), "to", test[DATE_COLUMN].max())

assert train[DATE_COLUMN].max() <= test[DATE_COLUMN].min()


In [ ]:
X_train = train[FEATURES].apply(pd.to_numeric, errors="coerce")
X_test = test[FEATURES].apply(pd.to_numeric, errors="coerce")
y_train = train[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

time_mae = mean_absolute_error(y_test, predictions)
time_rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("Time-aware MAE:", round(time_mae, 4))
print("Time-aware RMSE:", round(time_rmse, 4))


### Before / after comparison

The Week-5 metric must be copied from the executed Week-5 notebook. The code below deliberately refuses to invent that number.


In [ ]:
if WEEK5_MAE is None:
    print("WEEK5_MAE is not filled in. Copy the actual Week-5 MAE into the configuration cell.")
else:
    comparison = pd.DataFrame({
        "Evaluation": ["Week-5 original split", "Week-6 time-aware split"],
        "MAE": [WEEK5_MAE, time_mae]
    })
    display(comparison)


## 3. Leakage audit

The final feature set is screened for obvious leakage indicators such as future outcomes, later-period information, or labels. This is a screening check, not proof that leakage is impossible; feature definitions must also be reviewed manually.


In [ ]:
future_keywords = ["future", "next", "outcome", "label", "target", "after", "later"]

possible_leakage = [
    col for col in FEATURES
    if any(word in col.lower() for word in future_keywords)
]

print("Features checked:")
for col in FEATURES:
    print("-", col)

print("\nPotential leakage candidates:")
if possible_leakage:
    for col in possible_leakage:
        print("-", col)
else:
    print("None found from feature names.")

print("\nManual feature-definition review is still required.")


## 4. Claim rewrite

### Original claim

> The Random Forest model can identify which pages will improve after a content refresh.

### Safer claim

> On the evaluated dataset, the Random Forest model showed measured predictive performance for the selected outcome under the tested validation design. The result is directional decision-support evidence for prioritizing pages, not proof that a content refresh will cause improvement.

### Why I changed it

The original statement implied causation and future certainty. The evaluation measures model performance on historical data under a particular validation design. The revised claim describes what was observed and measured without claiming that the model proves a future refresh will succeed.


## Real failure examples

The largest prediction errors are useful diagnostics. They show where the model is less reliable, but they do not prove why those errors occurred.


In [ ]:
errors = test.copy()
errors["actual"] = y_test.to_numpy()
errors["predicted"] = predictions
errors["absolute_error"] = (errors["actual"] - errors["predicted"]).abs()

failure_examples = errors.sort_values("absolute_error", ascending=False).head(10)

display(
    failure_examples[
        FEATURES + ["actual", "predicted", "absolute_error"]
    ]
)


In [ ]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance)


### Error interpretation

The displayed observations are the largest absolute prediction errors in the time-aware test set. I will inspect them for unusual search demand, impressions, CTR, position, or other patterns.

These are observed and directional patterns, not causal explanations.


## Self-check

- [x] Every section contains markdown reasoning and supporting code.
- [x] Two research-paper findings have constructive methodology questions.
- [x] The model is evaluated using a time-aware split.
- [x] A before/after comparison is included.
- [x] The final feature set is checked for potential leakage.
- [x] Real failure examples are displayed.
- [x] The strongest claim is rewritten using safe language.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] No client names, URLs, or private queries are included.
- [ ] The notebook runs top-to-bottom after replacing the configuration placeholders with real Week-5 values.
- [ ] Saved as `work/notebooks/w06_validation_audit.ipynb` and committed to the repo.
